# SL_LSTM_SHAP: A-Share Price Prediction — Stacked LSTM + SHAP Explainability

**Course**: WM9B7 Artificial Intelligence & Deep Learning  
**Role**: AI Consultant at IntelliSys Ltd.  
**Task**: Build an A-share price prediction prototype for a financial sector client  
**Data**: CSI 300 Index (000300) daily OHLCV, 2019–2024  

| Module | Content |
|--------|---------|
| Data Acquisition | Multi-source: local cache → akshare(Sina) → akshare(Eastmoney) → yfinance |
| Feature Engineering | MA / RSI / MACD / BBands / Volume / A-share specific factors |
| Model | Stacked LSTM dual-head (regression + classification) |
| Evaluation | RMSE / MAE / R² / Accuracy / F1 |
| Explainability | SHAP KernelExplainer |

## 0. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, f1_score, classification_report
)

import akshare as ak
import shap

# ── 随机种子 ──────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── 输出目录 ──────────────────────────────────────────
OUT_DIR = Path('outputs')
OUT_DIR.mkdir(exist_ok=True)

## 1. Data Acquisition

Multi-source strategy (stops at first success):
1. **Local cache** — `outputs/hs300_20190101_20241231.csv`
2. **akshare** `index_zh_a_hist` (Sina Finance)
3. **akshare** `stock_zh_index_daily` (Eastmoney fallback)
4. **yfinance** `000300.SS` (Yahoo Finance)

In [ ]:
def _fmt_date(d8: str) -> str:
    return f'{d8[:4]}-{d8[4:6]}-{d8[6:]}'


def _post_process(df: pd.DataFrame, cache_path: Path, source: str) -> pd.DataFrame:
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    if 'turnover' not in df.columns:
        df['turnover'] = 0.0
    df['turnover'] = pd.to_numeric(df['turnover'], errors='coerce').fillna(0.0)
    if 'pct_change' not in df.columns:
        df['pct_change'] = df['close'].pct_change() * 100
    df['pct_change'] = pd.to_numeric(df['pct_change'], errors='coerce').fillna(0.0)
    for col in ['open', 'high', 'low', 'close', 'volume']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df.dropna(subset=['open', 'high', 'low', 'close', 'volume'], inplace=True)
    df.reset_index(drop=True, inplace=True)
    if source != 'local cache':
        df.to_csv(cache_path, index=False, encoding='utf-8-sig')
        print(f'Cached to {cache_path}')
    print(f'[{source}] {len(df)} rows  {df.date.min().date()} → {df.date.max().date()}')
    return df


def fetch_hs300(start: str = '20190101', end: str = '20241231') -> pd.DataFrame:
    cache_path = OUT_DIR / f'hs300_{start}_{end}.csv'

    # ── 优先：本地缓存 ────────────────────────────────
    if cache_path.exists():
        df = pd.read_csv(cache_path)
        return _post_process(df, cache_path, 'local cache')

    # ── 源1：akshare index_zh_a_hist（新浪）────────────
    try:
        df = ak.index_zh_a_hist(
            symbol='000300', period='daily',
            start_date=start, end_date=end
        )
        df.rename(columns={
            '日期': 'date', '开盘': 'open', '收盘': 'close',
            '最高': 'high', '最低': 'low', '成交量': 'volume',
            '成交额': 'amount', '振幅': 'amplitude',
            '涨跌幅': 'pct_change', '涨跌额': 'change', '换手率': 'turnover'
        }, inplace=True)
        return _post_process(df, cache_path, 'akshare/xinhua')
    except Exception as e:
        print(f'[akshare/xinhua] failed: {e}')

    # ── 源2：akshare stock_zh_index_daily（东财）────────
    try:
        df = ak.stock_zh_index_daily(symbol='sh000300')
        df['date'] = pd.to_datetime(df['date'])
        df = df[
            (df['date'] >= pd.Timestamp(_fmt_date(start))) &
            (df['date'] <= pd.Timestamp(_fmt_date(end)))
        ].copy()
        return _post_process(df, cache_path, 'akshare/eastmoney')
    except Exception as e:
        print(f'[akshare/eastmoney] failed: {e}')

    # ── 源3：yfinance 000300.SS ──────────────────────
    try:
        import yfinance as yf
        raw = yf.download('000300.SS', start=_fmt_date(start), end=_fmt_date(end),
                          progress=False, auto_adjust=True)
        if isinstance(raw.columns, pd.MultiIndex):
            raw.columns = raw.columns.get_level_values(0)
        raw = raw.reset_index()
        df = pd.DataFrame({
            'date': raw['Date'], 'open': raw['Open'], 'high': raw['High'],
            'low': raw['Low'], 'close': raw['Close'], 'volume': raw['Volume'],
        })
        return _post_process(df, cache_path, 'yfinance')
    except Exception as e:
        print(f'[yfinance] failed: {e}')

    raise RuntimeError(
        f'All sources failed and no local cache found at {cache_path}.\n'
        'Please manually place the CSV there with columns: date, open, high, low, close, volume'
    )


raw_df = fetch_hs300()
raw_df.head()

## 2. Feature Engineering

| Category | Features |
|----------|----------|
| Trend | MA5 / MA20 / MA60 |
| Momentum | RSI(14) / MACD / MACD Signal / MACD Hist |
| Volatility | Bollinger Upper / Middle / Lower |
| Volume | Volume change rate |
| A-share specific | Limit-up/down flag / Turnover rate |

In [ ]:
def add_ma(df: pd.DataFrame) -> pd.DataFrame:
    """移动平均线 MA5 / MA20 / MA60。"""
    df['ma5']  = df['close'].rolling(5).mean()
    df['ma20'] = df['close'].rolling(20).mean()
    df['ma60'] = df['close'].rolling(60).mean()
    return df


def add_rsi(df: pd.DataFrame, period: int = 14) -> pd.DataFrame:
    """RSI(14)。"""
    delta = df['close'].diff()
    gain  = delta.clip(lower=0).rolling(period).mean()
    loss  = (-delta.clip(upper=0)).rolling(period).mean()
    rs    = gain / (loss + 1e-9)
    df['rsi14'] = 100 - 100 / (1 + rs)
    return df


def add_macd(df: pd.DataFrame,
             fast: int = 12, slow: int = 26, signal: int = 9) -> pd.DataFrame:
    """MACD / Signal / Histogram。"""
    ema_fast          = df['close'].ewm(span=fast, adjust=False).mean()
    ema_slow          = df['close'].ewm(span=slow, adjust=False).mean()
    df['macd']        = ema_fast - ema_slow
    df['macd_signal'] = df['macd'].ewm(span=signal, adjust=False).mean()
    df['macd_hist']   = df['macd'] - df['macd_signal']
    return df


def add_bollinger(df: pd.DataFrame, period: int = 20, std: int = 2) -> pd.DataFrame:
    """布林带 Upper / Middle / Lower。"""
    mid             = df['close'].rolling(period).mean()
    sd              = df['close'].rolling(period).std()
    df['bb_upper']  = mid + std * sd
    df['bb_middle'] = mid
    df['bb_lower']  = mid - std * sd
    return df


def add_volume_change(df: pd.DataFrame) -> pd.DataFrame:
    """成交量变化率。"""
    df['vol_chg'] = df['volume'].pct_change().fillna(0)
    return df


def add_ashare_features(df: pd.DataFrame, limit: float = 9.5) -> pd.DataFrame:
    """A股特色因子：涨跌停近似标记（±limit%）、换手率。"""
    df['limit_up']   = (df['pct_change'] >=  limit).astype(int)
    df['limit_down'] = (df['pct_change'] <= -limit).astype(int)
    return df


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """串联所有因子，删除含 NaN 的行。"""
    df = df.copy()
    df = add_ma(df)
    df = add_rsi(df)
    df = add_macd(df)
    df = add_bollinger(df)
    df = add_volume_change(df)
    df = add_ashare_features(df)
    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df


feat_df = build_features(raw_df)
print(f'因子工程后数据量: {len(feat_df)} 行')
feat_df.tail(3)

## 3. Data Preprocessing

- `MinMaxScaler` normalisation (fit on train set only — no look-ahead bias)
- Sliding window: `window = 30`
- Labels: regression → next-day close (normalised); classification → Up / Flat / Down (threshold ±0.5%)
- Split ratio: 70% train / 15% val / 15% test

In [ ]:
FEATURE_COLS = [
    'open', 'high', 'low', 'close', 'volume',
    'ma5', 'ma20', 'ma60',
    'rsi14',
    'macd', 'macd_signal', 'macd_hist',
    'bb_upper', 'bb_middle', 'bb_lower',
    'vol_chg', 'turnover',
    'limit_up', 'limit_down'
]
WINDOW     = 30
CLS_THRESH = 0.5   # 涨跌判断阈值（%）

# ── 构建标签 ──────────────────────────────────────────
close_prices = feat_df['close'].values
next_close   = np.roll(close_prices, -1)              # 明日收盘（最后一行无效，后续会丢弃）
next_pct     = feat_df['pct_change'].shift(-1).values # 明日涨跌幅

def pct_to_label(pct: float) -> int:
    """0=跌, 1=平, 2=涨"""
    if pct > CLS_THRESH:  return 2
    if pct < -CLS_THRESH: return 0
    return 1

cls_labels = np.array([pct_to_label(p) for p in next_pct])

# ── 归一化 ────────────────────────────────────────────
n_total = len(feat_df)
n_train = int(n_total * 0.70)

feature_matrix = feat_df[FEATURE_COLS].values.astype(np.float32)
next_close_arr = next_close.astype(np.float32)

scaler_X     = MinMaxScaler()
scaler_close = MinMaxScaler()

feature_matrix[:n_train] = scaler_X.fit_transform(feature_matrix[:n_train])
feature_matrix[n_train:] = scaler_X.transform(feature_matrix[n_train:])

next_close_2d            = next_close_arr.reshape(-1, 1)
next_close_2d[:n_train]  = scaler_close.fit_transform(next_close_2d[:n_train])
next_close_2d[n_train:]  = scaler_close.transform(next_close_2d[n_train:])
next_close_norm          = next_close_2d.flatten()

# ── 滑动窗口切片 ──────────────────────────────────────
def make_sequences(features, reg_targets, cls_targets, window):
    X, y_reg, y_cls = [], [], []
    for i in range(len(features) - window):
        X.append(features[i : i + window])
        y_reg.append(reg_targets[i + window])
        y_cls.append(cls_targets[i + window])
    return (
        np.array(X, dtype=np.float32),
        np.array(y_reg, dtype=np.float32),
        np.array(y_cls, dtype=np.int64)
    )

X_all, y_reg_all, y_cls_all = make_sequences(
    feature_matrix, next_close_norm, cls_labels, WINDOW
)

# ── 7 : 1.5 : 1.5 划分 ───────────────────────────────
n    = len(X_all)
n_tr = int(n * 0.70)
n_va = int(n * 0.15)

X_train, y_reg_train, y_cls_train = X_all[:n_tr],           y_reg_all[:n_tr],           y_cls_all[:n_tr]
X_val,   y_reg_val,   y_cls_val   = X_all[n_tr:n_tr+n_va],  y_reg_all[n_tr:n_tr+n_va],  y_cls_all[n_tr:n_tr+n_va]
X_test,  y_reg_test,  y_cls_test  = X_all[n_tr+n_va:],      y_reg_all[n_tr+n_va:],      y_cls_all[n_tr+n_va:]

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')
print(f'分类标签分布: 跌={np.sum(y_cls_all==0)}, 平={np.sum(y_cls_all==1)}, 涨={np.sum(y_cls_all==2)}')

In [ ]:
class StockDataset(Dataset):
    def __init__(self, X, y_reg, y_cls):
        self.X     = torch.from_numpy(X)
        self.y_reg = torch.from_numpy(y_reg).unsqueeze(1)
        self.y_cls = torch.from_numpy(y_cls)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y_reg[idx], self.y_cls[idx]


BATCH_SIZE = 64

train_loader = DataLoader(StockDataset(X_train, y_reg_train, y_cls_train),
                          batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader   = DataLoader(StockDataset(X_val,   y_reg_val,   y_cls_val),
                          batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(StockDataset(X_test,  y_reg_test,  y_cls_test),
                          batch_size=BATCH_SIZE, shuffle=False)

## 4. Model — Stacked LSTM (Dual Head)

```
Input  (batch, 30, n_features)
  └─ LSTM × 2 layers, hidden=128, dropout=0.3
       └─ last hidden state
            ├─ Regression head  → 1   (next-day close, normalised)
            └─ Classification head → 3  (Down / Flat / Up)
```

In [ ]:
class StackedLSTM(nn.Module):
    def __init__(self, input_size: int, hidden_size: int = 64,
                 num_layers: int = 2, dropout: float = 0.5,
                 num_classes: int = 3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            batch_first=True
        )
        self.dropout  = nn.Dropout(dropout)
        self.reg_head = nn.Linear(hidden_size, 1)
        self.cls_head = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _  = self.lstm(x)
        out     = self.dropout(out[:, -1, :])
        reg_out = self.reg_head(out)
        cls_out = self.cls_head(out)
        return reg_out, cls_out


N_FEATURES = len(FEATURE_COLS)
model = StackedLSTM(input_size=N_FEATURES).to(DEVICE)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTrainable parameters: {total_params:,}')

## 5. Training

### 5a. Time-Series Cross-Validation (TimeSeriesSplit, k=5)

Standard KFold would leak future data into training. `TimeSeriesSplit` keeps temporal order intact — each fold's val set is always **after** its train set.

### 5b. Final Training on Full Train Set

In [ ]:
EPOCHS          = 50
LR              = 1e-3
WEIGHT_DECAY    = 1e-5
LOSS_ALPHA      = 0.5
EARLY_STOP_PAT  = 10   # early stopping patience

optimizer     = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler     = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

# Class weights to handle imbalance (down / flat / up)
cls_counts  = np.bincount(y_cls_train)
cls_weights = torch.FloatTensor(1.0 / cls_counts).to(DEVICE)
cls_weights = cls_weights / cls_weights.sum() * len(cls_counts)
print(f'Class counts (train): down={cls_counts[0]}, flat={cls_counts[1]}, up={cls_counts[2]}')
print(f'Class weights: {cls_weights.cpu().numpy().round(3)}')

reg_criterion = nn.MSELoss()
cls_criterion = nn.CrossEntropyLoss(weight=cls_weights)


def run_epoch(loader, train: bool):
    model.train() if train else model.eval()
    total_loss = total_reg = total_cls = 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for X_b, y_reg_b, y_cls_b in loader:
            X_b, y_reg_b, y_cls_b = X_b.to(DEVICE), y_reg_b.to(DEVICE), y_cls_b.to(DEVICE)
            reg_out, cls_out = model(X_b)
            loss_reg = reg_criterion(reg_out, y_reg_b)
            loss_cls = cls_criterion(cls_out, y_cls_b)
            loss     = LOSS_ALPHA * loss_reg + (1 - LOSS_ALPHA) * loss_cls
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            total_loss += loss.item()
            total_reg  += loss_reg.item()
            total_cls  += loss_cls.item()
    n = len(loader)
    return total_loss / n, total_reg / n, total_cls / n


history = {k: [] for k in
           ['train_loss','val_loss','train_reg','val_reg','train_cls','val_cls']}
best_val_loss   = float('inf')
best_state      = None
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_reg, tr_cls = run_epoch(train_loader, train=True)
    va_loss, va_reg, va_cls = run_epoch(val_loader,   train=False)
    scheduler.step(va_loss)

    for k, v in zip(history, [tr_loss, va_loss, tr_reg, va_reg, tr_cls, va_cls]):
        history[k].append(v)

    if va_loss < best_val_loss:
        best_val_loss    = va_loss
        best_state       = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS} | '
              f'Train {tr_loss:.4f} (r={tr_reg:.4f} c={tr_cls:.4f}) | '
              f'Val   {va_loss:.4f} (r={va_reg:.4f} c={va_cls:.4f}) | '
              f'Patience {patience_counter}/{EARLY_STOP_PAT}')

    if patience_counter >= EARLY_STOP_PAT:
        print(f'\nEarly stopping at epoch {epoch} (no improvement for {EARLY_STOP_PAT} epochs)')
        break

model.load_state_dict(best_state)
print(f'\nBest model restored (val_loss={best_val_loss:.4f})')

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

N_SPLITS    = 5
EPOCHS_CV   = 30   # fewer epochs per fold to keep runtime reasonable
LOSS_ALPHA  = 0.5

tscv = TimeSeriesSplit(n_splits=N_SPLITS)

# CV runs on the combined train+val set (X_all[:n_tr+n_va])
X_cv     = X_all[:n_tr + n_va]
y_reg_cv = y_reg_all[:n_tr + n_va]
y_cls_cv = y_cls_all[:n_tr + n_va]

cv_results = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_cv), 1):
    print(f'\n── Fold {fold}/{N_SPLITS}  train={len(train_idx)}  val={len(val_idx)} ──')

    # Build fold datasets
    fold_train = DataLoader(
        StockDataset(X_cv[train_idx], y_reg_cv[train_idx], y_cls_cv[train_idx]),
        batch_size=BATCH_SIZE, shuffle=False, drop_last=True
    )
    fold_val = DataLoader(
        StockDataset(X_cv[val_idx], y_reg_cv[val_idx], y_cls_cv[val_idx]),
        batch_size=BATCH_SIZE, shuffle=False
    )

    # Fresh model for each fold
    fold_model = StackedLSTM(input_size=N_FEATURES).to(DEVICE)
    fold_opt   = torch.optim.Adam(fold_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    fold_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(fold_opt, mode='min', factor=0.5, patience=3)

    cls_counts_fold  = np.bincount(y_cls_cv[train_idx])
    cls_weights_fold = torch.FloatTensor(1.0 / cls_counts_fold).to(DEVICE)
    cls_weights_fold = cls_weights_fold / cls_weights_fold.sum() * len(cls_counts_fold)

    reg_crit = nn.MSELoss()
    cls_crit = nn.CrossEntropyLoss(weight=cls_weights_fold)

    best_fold_val = float('inf')
    best_fold_state = None

    for epoch in range(1, EPOCHS_CV + 1):
        # ── train ──
        fold_model.train()
        for X_b, y_reg_b, y_cls_b in fold_train:
            X_b, y_reg_b, y_cls_b = X_b.to(DEVICE), y_reg_b.to(DEVICE), y_cls_b.to(DEVICE)
            reg_out, cls_out = fold_model(X_b)
            loss = LOSS_ALPHA * reg_crit(reg_out, y_reg_b) + (1 - LOSS_ALPHA) * cls_crit(cls_out, y_cls_b)
            fold_opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(fold_model.parameters(), 1.0)
            fold_opt.step()

        # ── val ──
        fold_model.eval()
        val_loss = val_reg = val_cls_l = 0.0
        all_reg_pred, all_reg_true, all_cls_pred, all_cls_true = [], [], [], []
        with torch.no_grad():
            for X_b, y_reg_b, y_cls_b in fold_val:
                X_b, y_reg_b, y_cls_b = X_b.to(DEVICE), y_reg_b.to(DEVICE), y_cls_b.to(DEVICE)
                reg_out, cls_out = fold_model(X_b)
                val_loss  += (LOSS_ALPHA * reg_crit(reg_out, y_reg_b) +
                              (1-LOSS_ALPHA) * cls_crit(cls_out, y_cls_b)).item()
                val_reg   += reg_crit(reg_out, y_reg_b).item()
                val_cls_l += cls_crit(cls_out, y_cls_b).item()
                all_reg_pred.append(reg_out.cpu().numpy())
                all_reg_true.append(y_reg_b.cpu().numpy())
                all_cls_pred.append(cls_out.argmax(1).cpu().numpy())
                all_cls_true.append(y_cls_b.cpu().numpy())

        n_batches = len(fold_val)
        fold_sched.step(val_loss / n_batches)
        if val_loss < best_fold_val:
            best_fold_val   = val_loss
            best_fold_state = {k: v.cpu().clone() for k, v in fold_model.state_dict().items()}

    # ── fold metrics ──
    fold_model.load_state_dict(best_fold_state)
    pred_r = scaler_close.inverse_transform(np.concatenate(all_reg_pred).reshape(-1,1)).flatten()
    true_r = scaler_close.inverse_transform(np.concatenate(all_reg_true).reshape(-1,1)).flatten()
    pred_c = np.concatenate(all_cls_pred)
    true_c = np.concatenate(all_cls_true)

    rmse = np.sqrt(mean_squared_error(true_r, pred_r))
    mae  = mean_absolute_error(true_r, pred_r)
    r2   = r2_score(true_r, pred_r)
    acc  = accuracy_score(true_c, pred_c)
    f1   = f1_score(true_c, pred_c, average='macro', zero_division=0)

    cv_results.append({'fold': fold, 'RMSE': rmse, 'MAE': mae, 'R2': r2, 'Acc': acc, 'F1': f1})
    print(f'  RMSE={rmse:.2f}  MAE={mae:.2f}  R²={r2:.4f}  Acc={acc:.4f}  F1={f1:.4f}')

# ── summary ──
cv_df = pd.DataFrame(cv_results).set_index('fold')
print('\n── Cross-Validation Summary ──')
print(cv_df.to_string())
print('\nMean ± Std:')
print(cv_df.agg(['mean','std']).round(4).to_string())

## 6. Evaluation

- **Regression**: RMSE, MAE, R²
- **Classification**: Accuracy, F1 (macro)
- **Visualisation**: learning curves, predicted vs actual price

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ep = range(1, len(history['train_loss']) + 1)  # actual epochs run (respects early stopping)
for ax, (tr_k, va_k), title in zip(
    axes,
    [('train_loss','val_loss'), ('train_reg','val_reg'), ('train_cls','val_cls')],
    ['Total Loss', 'Regression Loss (MSE)', 'Classification Loss (CE)']
):
    ax.plot(ep, history[tr_k], label='Train')
    ax.plot(ep, history[va_k], label='Val')
    ax.set_title(title); ax.set_xlabel('Epoch')
    ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('Learning Curves — Stacked LSTM', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/learning_curves.png')

In [ ]:
def collect_predictions(loader):
    model.eval()
    all_reg, all_cls, all_y_reg, all_y_cls = [], [], [], []
    with torch.no_grad():
        for X_b, y_reg_b, y_cls_b in loader:
            reg_out, cls_out = model(X_b.to(DEVICE))
            all_reg.append(reg_out.cpu().numpy())
            all_cls.append(cls_out.argmax(dim=1).cpu().numpy())
            all_y_reg.append(y_reg_b.numpy())
            all_y_cls.append(y_cls_b.numpy())
    return (
        np.concatenate(all_reg).flatten(),
        np.concatenate(all_cls),
        np.concatenate(all_y_reg).flatten(),
        np.concatenate(all_y_cls)
    )


pred_reg, pred_cls, true_reg, true_cls = collect_predictions(test_loader)

pred_reg_inv = scaler_close.inverse_transform(pred_reg.reshape(-1, 1)).flatten()
true_reg_inv = scaler_close.inverse_transform(true_reg.reshape(-1, 1)).flatten()

rmse = np.sqrt(mean_squared_error(true_reg_inv, pred_reg_inv))
mae  = mean_absolute_error(true_reg_inv, pred_reg_inv)
r2   = r2_score(true_reg_inv, pred_reg_inv)
acc  = accuracy_score(true_cls, pred_cls)
f1   = f1_score(true_cls, pred_cls, average='macro')

print('=' * 45)
print('【回归指标 — 测试集】')
print(f'  RMSE : {rmse:.4f}')
print(f'  MAE  : {mae:.4f}')
print(f'  R²   : {r2:.4f}')
print()
print('【分类指标 — 测试集】')
print(f'  Accuracy  : {acc:.4f}')
print(f'  F1 (macro): {f1:.4f}')
print('=' * 45)
print()
print(classification_report(true_cls, pred_cls, target_names=['跌', '平', '涨']))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(true_reg_inv, label='Actual Close', linewidth=1.2)
ax.plot(pred_reg_inv, label='Predicted Close', linewidth=1.2, alpha=0.8)
ax.set_title('CSI 300 — Predicted vs Actual Close Price (Test Set)', fontsize=13)
ax.set_xlabel('Time Step')
ax.set_ylabel('Close Price (pts)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'prediction_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/prediction_vs_actual.png')

## 7. SHAP Explainability

`KernelExplainer` is used to analyse each feature's contribution to the classification output.  
Only the **last time step** of each sample is passed as SHAP input to avoid dimensionality explosion.

In [ ]:
def predict_proba_last_step(X_last: np.ndarray) -> np.ndarray:
    """SHAP 包装：将最后时间步特征复制为完整序列后推理，返回 softmax 概率。"""
    X_seq  = np.tile(X_last[:, np.newaxis, :], (1, WINDOW, 1)).astype(np.float32)
    tensor = torch.from_numpy(X_seq).to(DEVICE)
    model.eval()
    with torch.no_grad():
        _, cls_out = model(tensor)
        return torch.softmax(cls_out, dim=1).cpu().numpy()


np.random.seed(SEED)
bg_idx    = np.random.choice(len(X_train), size=100, replace=False)
X_bg      = X_train[bg_idx, -1, :]    # (100, n_features)
X_explain = X_test[:50,   -1, :]      # (50,  n_features)

print('初始化 SHAP KernelExplainer...')
explainer   = shap.KernelExplainer(predict_proba_last_step, X_bg)

print('计算 SHAP 值（nsamples=200，约需数分钟）...')
shap_values = explainer.shap_values(X_explain, nsamples=200)
print('SHAP 计算完成。')

In [ ]:
CLASS_IDX   = 2   # 0=down, 1=flat, 2=up
CLASS_LABEL = ['Down', 'Flat', 'Up'][CLASS_IDX]

# Compatible with both old (list) and new (ndarray) SHAP formats
if isinstance(shap_values, list):
    sv = shap_values[CLASS_IDX]
elif shap_values.ndim == 3:
    sv = shap_values[:, :, CLASS_IDX]
else:
    sv = shap_values

shap.summary_plot(
    sv,
    X_explain,
    feature_names=FEATURE_COLS,
    plot_type='dot',
    show=False,
    max_display=len(FEATURE_COLS)
)
plt.title(f'SHAP Summary Plot — Class: {CLASS_LABEL}', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / 'shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/shap_summary.png')

In [ ]:
if isinstance(shap_values, list):
    mean_abs_shap = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
elif shap_values.ndim == 3:
    mean_abs_shap = np.abs(shap_values).mean(axis=(0, 2))
else:
    mean_abs_shap = np.abs(shap_values).mean(axis=0)

importance_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=True)

fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(importance_df['feature'], importance_df['mean_abs_shap'],
        color='steelblue', edgecolor='white')
ax.set_title('Mean |SHAP| Feature Importance (All Classes)', fontsize=13)
ax.set_xlabel('Mean |SHAP value|')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'shap_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/shap_feature_importance.png')

print('\nTop 5 features:')
print(importance_df.sort_values('mean_abs_shap', ascending=False).head(5).to_string(index=False))

## 8. Summary

| Output | Path |
|--------|------|
| Learning curves | `outputs/learning_curves.png` |
| Predicted vs Actual | `outputs/prediction_vs_actual.png` |
| SHAP Summary | `outputs/shap_summary.png` |
| SHAP Feature Importance | `outputs/shap_feature_importance.png` |

### Limitations & Future Work
- **Data**: Index-level data used; individual A-shares require handling disclosure delays
- **Features**: Sentiment factors (NLP on news/social media) could be added
- **Model**: Compare against Transformer or TCN architectures
- **SHAP**: Replace KernelExplainer with DeepExplainer for faster computation